# SFT: messy text → strict JSON extraction

This notebook is a **thin demo**: every step calls into the tested `geap_tuning`
package. It mirrors [`examples/run_sft_extraction.py`](../examples/run_sft_extraction.py).

Our other SFT demos are all **classification** (support-intent, banking77,
oral-disease images). This one is **generative structured output**: rewrite a
messy natural-language order line into a strict JSON object with exactly five
keys. It shows a **before → after** lift on an objective field-exact-match
metric — the untuned base tends to add prose / code fences and emit `quantity`
as a string (`"3"` not `3`), so there is real headroom.

> **Requires live GCP and incurs tuning cost** (one SFT job). Have a real `.env`
> and `gcloud auth` in place.

In [ ]:
from geap_tuning.config import genai_client, load_config

BASE_MODEL = "gemini-2.5-flash"
cfg = load_config()
client = genai_client(cfg)
cfg

## 1. Build the dataset and stage it to GCS

The bank is generated deterministically (correct-by-construction): each messy
input line has a matching gold JSON object, with `quantity` kept an **int** so a
string answer is genuinely wrong. The `SYSTEM_INSTRUCTION` demands only the JSON
object — no prose, no fences.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.sft.extraction import (
    EXTRACTION_EXAMPLES,
    SYSTEM_INSTRUCTION,
    build_extraction_dataset,
    build_records,
    split_dataset,
)

paths = build_extraction_dataset("../datasets/sft_json_extraction")
train_uri = upload_file(paths["train"], f"{cfg.bucket}/sft_json_extraction/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/sft_json_extraction/val.jsonl")

_, _, test = split_dataset(EXTRACTION_EXAMPLES)
test_records = build_records(test)
print(f"{len(EXTRACTION_EXAMPLES)} examples, {len(test_records)} held out")
print(SYSTEM_INSTRUCTION)

## 2. Score the untuned base (the "before")

`run_eval` parses each reply as JSON (stripping fences / surrounding prose) and
scores micro field-exact-match `accuracy` plus `json_validity` and whole-object
`exact_match`. Comparison is type-insensitive only where it should not be — the
gold `quantity` is an int, so `"3"` still counts, but the base often drifts.

In [ ]:
from geap_tuning.inference import generate
from geap_tuning.sft.extraction_eval import run_eval

base = run_eval(
    test_records,
    predict_fn=lambda t: generate(client, BASE_MODEL, t, system_instruction=SYSTEM_INSTRUCTION),
)
print(
    f"BASE accuracy={base['accuracy']:.3f} exact_match={base['exact_match']:.3f} "
    f"json_validity={base['json_validity']:.3f}"
)

## 3. Launch the SFT job and wait

Reuse an existing job with the same display name if one exists (cost control);
otherwise launch a fresh supervised job on the same `client.tunings.tune(...)`
call with `method="SUPERVISED"`.

In [ ]:
from geap_tuning.jobs import find_tuning_job_by_display_name, tuned_endpoint, wait_for_tuning_job
from geap_tuning.sft.tune import launch_sft_job

DISPLAY_NAME = "geap-sft-json-extraction"

job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_sft_job(
        client,
        train_uri=train_uri,
        val_uri=val_uri,
        display_name=DISPLAY_NAME,
        base_model=BASE_MODEL,
        labels=cfg.labels,
    )
job = wait_for_tuning_job(client, job.name)
endpoint = tuned_endpoint(job)
endpoint

## 4. Score the tuned endpoint (the "after") and report the lift

In [ ]:
tuned = run_eval(
    test_records,
    predict_fn=lambda t: generate(client, endpoint, t, system_instruction=SYSTEM_INSTRUCTION),
)
print(
    f"TUNED accuracy={tuned['accuracy']:.3f} exact_match={tuned['exact_match']:.3f} "
    f"json_validity={tuned['json_validity']:.3f}"
)
print(
    f"LIFT accuracy +{tuned['accuracy'] - base['accuracy']:.3f}; "
    f"exact_match +{tuned['exact_match'] - base['exact_match']:.3f}; "
    f"json_validity {base['json_validity']:.3f}->{tuned['json_validity']:.3f}"
)